# GLAMMAR AI — Wan 2.1 1.3B

Open text-to-video model.

1. Sign into Google. Choose **Runtime → Change runtime type → T4 GPU** (free GPUs are not guaranteed).
2. Choose **Runtime → Run all**, approve after reviewing, then use the controls at the bottom.

Generation takes several minutes on a free T4 and may run out of memory.

Weights download into this temporary session. Save results before it disconnects. This notebook runs interactively in Colab — it is not an API server. Free Colab policies: https://research.google.com/colaboratory/faq.html

In [ ]:
%pip -q install diffusers transformers accelerate safetensors sentencepiece protobuf imageio[ffmpeg] ipywidgets
print("Dependencies installed. Continue below.")

In [ ]:
import torch
if not torch.cuda.is_available():
    raise RuntimeError("No GPU: choose Runtime > Change runtime type > T4 GPU, then Runtime > Run all.")
print("GPU:", torch.cuda.get_device_name(0))

import ipywidgets as w
from IPython.display import display, clear_output, Video
from google.colab import output, userdata
from diffusers.utils import export_to_video
from diffusers import WanPipeline
output.enable_custom_widget_manager()

try:
    TOKEN = userdata.get("HF_TOKEN")
except Exception:
    TOKEN = None

pipe = WanPipeline.from_pretrained(REPO, token=TOKEN, torch_dtype=torch.float16)
pipe.enable_model_cpu_offload()
print("Pipeline ready:", "Wan-AI/Wan2.1-T2V-1.3B-Diffusers")

prompt = w.Textarea(placeholder="Describe the video (short, single scene works best)…", layout=w.Layout(width="100%", height="110px"))
send = w.Button(description="Generate video", button_style="primary", icon="play")
log = w.Output()

def on_send(_):
    send.disabled = True
    with log:
        clear_output(wait=True)
        try:
            text = prompt.value.strip()[:1000]
            if not text:
                raise ValueError("Describe the video first.")
            with torch.inference_mode():
                frames = pipe(prompt=text, negative_prompt="low quality", num_frames=49, guidance_scale=5.0, height=320, width=512).frames[0]
            export_to_video(frames, "/content/glammar-video.mp4", fps=8)
            display(Video("/content/glammar-video.mp4", embed=True))
            print("Saved in the Colab Files panel: glammar-video.mp4")
        except Exception as e:
            print("Could not complete:", e)
            print("Video models need a lot of GPU memory. If you see out-of-memory errors, restart the runtime or use the GLAMMAR cloud server instead.")
        finally:
            send.disabled = False

send.on_click(on_send)
display(w.VBox([prompt, send, log]))